# fiftyone_review_processed.ipynb — browse CONVERTED (intermediate-schema) data

**When to use this:** *after* Stage 5.2 conversion. Loads a converted source's intermediate-schema output (DEC-046) directly — flat `images/`+`labels/`, canonical class ids, plain YOLO `.txt` labels, no train/val/test split yet. This is the stage Stage 5.3 (Box Audit) and Stage 5.5 (Model-Assisted Curation) actually work on, so this notebook is genuinely useful, not just a sanity check.

**Why not FiftyOne's built-in YOLO importer:** `fo.types.YOLOv5Dataset` assumes a `dataset.yaml` + per-split (`train`/`val`/`test`) folder structure, which this schema deliberately doesn't have yet (splits are only computed once, at Stage 5.8 — DEC-036). Rather than faking that structure, this notebook builds the FiftyOne dataset directly from `images/`+`labels/`.

**Not for:** raw acquisition-stage exports (`dataset/raw/<source>/`) — use `fiftyone_explore.ipynb` (COCO-style) or `fiftyone_preview.ipynb` (pre-pull) for those instead.

In [ ]:
# Imports
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk up from `start` to find the repo root (has config/ + AGENTS.md).

    Needed because notebooks live in notebooks/, not the repo root, and
    Jupyter's working directory depends on how it was launched -- this
    makes the scripts.* import below robust regardless of that.
    """
    for parent in [start, *start.parents]:
        if (parent / "config").is_dir() and (parent / "AGENTS.md").is_file():
            return parent
    raise RuntimeError("Could not locate repo root from notebook cwd.")


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

import fiftyone as fo

from scripts.utils.config_loader import get_canonical_names
from scripts.utils.file_utils import processed_dir

CANONICAL_NAMES = get_canonical_names()

In [ ]:
# Change this and re-run the cells below to browse a different processed
# source. Matches dataset/processed/<source_key>/ -- e.g. "exdark",
# "dataset_ninja_pothole_detection", "dataset_ninja_road_damage_detector",
# "open_images", or any "roboflow_<project_key>" (e.g. "roboflow_pothole_vhmow").
source_key = "dataset_ninja_pothole_detection"

In [ ]:
# Build the FiftyOne dataset directly from images/+labels/ -- no network
# calls, safe to re-run anytime. Each run replaces the same throwaway
# dataset (nothing persisted, nothing written back to disk).
images_dir = processed_dir(source_key) / "images"
labels_dir = processed_dir(source_key) / "labels"

dataset_name = f"review_{source_key}"
if dataset_name in fo.list_datasets():
    fo.delete_dataset(dataset_name)
dataset = fo.Dataset(dataset_name, persistent=False)

samples = []
for image_path in sorted(images_dir.iterdir()):
    label_path = labels_dir / f"{image_path.stem}.txt"
    sample = fo.Sample(filepath=str(image_path))

    detections = []
    if label_path.is_file():
        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            class_id = int(parts[0])
            cx, cy, w, h = (float(v) for v in parts[1:5])
            # Our labels are YOLO center-based (cx, cy, w, h); FiftyOne's
            # Detection.bounding_box is top-left-based (x, y, w, h) --
            # both normalized [0, 1], so just shift the origin.
            x, y = cx - w / 2, cy - h / 2
            detections.append(
                fo.Detection(label=CANONICAL_NAMES[class_id], bounding_box=[x, y, w, h])
            )

    sample["ground_truth"] = fo.Detections(detections=detections)
    samples.append(sample)

dataset.add_samples(samples)
print(f"{len(dataset)} images loaded from {images_dir}")

In [ ]:
# Launch the App
session = fo.launch_app(dataset, auto=False)

In [ ]:
session